In [19]:
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer

In [20]:
# loading the data set 
df = pd.read_csv("../data/twitter_training.csv")
df

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...
...,...,...,...,...
74676,9200,Nvidia,Positive,Just realized that the Windows partition of my...
74677,9200,Nvidia,Positive,Just realized that my Mac window partition is ...
74678,9200,Nvidia,Positive,Just realized the windows partition of my Mac ...
74679,9200,Nvidia,Positive,Just realized between the windows partition of...


### preprocessing 

In [21]:
# column cleaning 
df.columns

Index(['2401', 'Borderlands', 'Positive',
       'im getting on borderlands and i will murder you all ,'],
      dtype='str')

In [22]:
# drop the 2401 columns it irrelevant 
df.drop(columns=["2401"], inplace=True)
df

,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,Borderlands,Positive,I am coming to the borders and I will kill you...
1,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,Borderlands,Positive,im coming on borderlands and i will murder you...
3,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,Borderlands,Positive,im getting into borderlands and i can murder y...
...,...,...,...
74676,Nvidia,Positive,Just realized that the Windows partition of my...
74677,Nvidia,Positive,Just realized that my Mac window partition is ...
74678,Nvidia,Positive,Just realized the windows partition of my Mac ...
74679,Nvidia,Positive,Just realized between the windows partition of...


In [23]:
# rename columns 
df.rename(columns={
    'Borderlands':'game_name',
    'Positive':'sentiment',
    'im getting on borderlands and i will murder you all ,':'tweet'

},inplace=True)

df.columns

Index(['game_name', 'sentiment', 'tweet'], dtype='str')

In [24]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 74681 entries, 0 to 74680
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   game_name  74681 non-null  str  
 1   sentiment  74681 non-null  str  
 2   tweet      73995 non-null  str  
dtypes: str(3)
memory usage: 1.7 MB


In [25]:
# drop null values 
df.isna().sum()
df.dropna(inplace=True)
df.isna().sum()

game_name    0
sentiment    0
tweet        0
dtype: int64

In [26]:
# check for duplicates 
df.duplicated().sum()
df.drop_duplicates(inplace=True)
df.duplicated().sum()

np.int64(0)

In [27]:
df.shape

(70957, 3)

In [28]:
# ensure balance between sentiment by reducing trimming data
df["sentiment"].value_counts()

sentiment
Negative      21565
Positive      19548
Neutral       17398
Irrelevant    12446
Name: count, dtype: int64

In [29]:
df_negative = df[df.sentiment == 'Negative'].iloc[:2000]
df_positive = df[df.sentiment == 'Positive'].iloc[:2000]
df_neutral = df[df.sentiment == 'Neutral'].iloc[:2000]
df_irrelevant = df[df.sentiment == 'Irrelevant'].iloc[:2000]

In [30]:
# now our df has 8000 dataset 
df = pd.concat([df_negative,df_positive,df_positive,df_irrelevant])
df

,game_name,sentiment,tweet
23,Borderlands,Negative,the biggest dissappoinment in my life came out...
24,Borderlands,Negative,The biggest disappointment of my life came a y...
26,Borderlands,Negative,the biggest dissappoinment in my life coming o...
27,Borderlands,Negative,For the biggest male dissappoinment in my life...
28,Borderlands,Negative,the biggest dissappoinment in my life came bac...
...,...,...,...
10788,Xbox(Xseries),Irrelevant,Latest news about Xbox Geek! paper.li / XboxGe...
10789,Xbox(Xseries),Irrelevant,Latest xbox geek! paper.li / XboxGeekNews / 1....
10790,Xbox(Xseries),Irrelevant,The latest xbox hack blog! paper.li/XboxGeekNe...
10791,Xbox(Xseries),Irrelevant,The latest xbox from geek news! smart paper. l...


In [31]:
# categorical to numeric 
df_game = pd.get_dummies(df['game_name']).astype("int")
df_game

,Amazon,Borderlands,CallOfDutyBlackopsColdWar,Overwatch,Xbox(Xseries)
23,0,1,0,0,0
24,0,1,0,0,0
26,0,1,0,0,0
27,0,1,0,0,0
28,0,1,0,0,0
...,...,...,...,...,...
10788,0,0,0,0,1
10789,0,0,0,0,1
10790,0,0,0,0,1
10791,0,0,0,0,1


In [32]:
df.drop(columns=["game_name"],inplace=True)
df

,sentiment,tweet
23,Negative,the biggest dissappoinment in my life came out...
24,Negative,The biggest disappointment of my life came a y...
26,Negative,the biggest dissappoinment in my life coming o...
27,Negative,For the biggest male dissappoinment in my life...
28,Negative,the biggest dissappoinment in my life came bac...
...,...,...
10788,Irrelevant,Latest news about Xbox Geek! paper.li / XboxGe...
10789,Irrelevant,Latest xbox geek! paper.li / XboxGeekNews / 1....
10790,Irrelevant,The latest xbox hack blog! paper.li/XboxGeekNe...
10791,Irrelevant,The latest xbox from geek news! smart paper. l...


In [33]:
nlp = spacy.load("en_core_web_md")


In [34]:
def lemmatization(text):
    doc = nlp(text)
    lemmaList = [word.lemma_ for word in doc]
    return ' '.join(lemmaList)

df["lemma"] = df["tweet"].apply(lemmatization)
df

,sentiment,tweet,lemma
23,Negative,the biggest dissappoinment in my life came out...,the big dissappoinment in my life come out a y...
24,Negative,The biggest disappointment of my life came a y...,the big disappointment of my life come a year ...
26,Negative,the biggest dissappoinment in my life coming o...,the big dissappoinment in my life come out a y...
27,Negative,For the biggest male dissappoinment in my life...,for the big male dissappoinment in my life com...
28,Negative,the biggest dissappoinment in my life came bac...,the big dissappoinment in my life come back la...
...,...,...,...
10788,Irrelevant,Latest news about Xbox Geek! paper.li / XboxGe...,late news about Xbox Geek ! paper.li / XboxGee...
10789,Irrelevant,Latest xbox geek! paper.li / XboxGeekNews / 1....,late xbox geek ! paper.li / XboxGeekNews / 1 ....
10790,Irrelevant,The latest xbox hack blog! paper.li/XboxGeekNe...,the late xbox hack blog ! paper.li/XboxGeekNew...
10791,Irrelevant,The latest xbox from geek news! smart paper. l...,the late xbox from geek news ! smart paper . l...


In [35]:
def remove_stopwords(text):
    doc = nlp(text)
    no_stopwords = [word.text for word in doc if not word.is_stop and not word.is_punct]
    return ' '.join(no_stopwords)

df['final_tweet'] = df["lemma"].apply(remove_stopwords)
df.drop(columns=["lemma","tweet"],inplace=True)
df

,sentiment,final_tweet
23,Negative,big dissappoinment life come year ago fuck bor...
24,Negative,big disappointment life come year ago
26,Negative,big dissappoinment life come year ago fuck bor...
27,Negative,big male dissappoinment life come hang year ti...
28,Negative,big dissappoinment life come year ago fuck bor...
...,...,...
10788,Irrelevant,late news Xbox Geek paper.li XboxGeekNews 1 th...
10789,Irrelevant,late xbox geek paper.li XboxGeekNews 1 thank c...
10790,Irrelevant,late xbox hack blog paper.li/XboxGeekNews/1 th...
10791,Irrelevant,late xbox geek news smart paper li xboxgeeknew...


In [40]:
df = pd.concat([df_game,df],axis=1)
df

,Amazon,Borderlands,CallOfDutyBlackopsColdWar,Overwatch,Xbox(Xseries),Amazon,Borderlands,CallOfDutyBlackopsColdWar,Overwatch,Xbox(Xseries),sentiment,final_tweet
23,0,1,0,0,0,0,1,0,0,0,Negative,big dissappoinment life come year ago fuck bor...
24,0,1,0,0,0,0,1,0,0,0,Negative,big disappointment life come year ago
26,0,1,0,0,0,0,1,0,0,0,Negative,big dissappoinment life come year ago fuck bor...
27,0,1,0,0,0,0,1,0,0,0,Negative,big male dissappoinment life come hang year ti...
28,0,1,0,0,0,0,1,0,0,0,Negative,big dissappoinment life come year ago fuck bor...
...,...,...,...,...,...,...,...,...,...,...,...,...
10788,0,0,0,0,1,0,0,0,0,1,Irrelevant,late news Xbox Geek paper.li XboxGeekNews 1 th...
10789,0,0,0,0,1,0,0,0,0,1,Irrelevant,late xbox geek paper.li XboxGeekNews 1 thank c...
10790,0,0,0,0,1,0,0,0,0,1,Irrelevant,late xbox hack blog paper.li/XboxGeekNews/1 th...
10791,0,0,0,0,1,0,0,0,0,1,Irrelevant,late xbox geek news smart paper li xboxgeeknew...


In [42]:
x = df.drop(columns=["sentiment"])
y = df["sentiment"]

y.shape,x.shape

((8000,), (8000, 11))

In [ ]:
#
tfidf = TfidfVectorizer()
tfidf.fit_transform